# Functional connections
Training-only DE similarity and input-specific PLV graphs; PLV uses only the current 30-second trial.

In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'
import json,time,hashlib
from pathlib import Path
import numpy as np
from scipy.signal import butter,sosfiltfilt,hilbert
root=Path(r'D:\EEG Experiment Artifacts\2026-11-02\graph_experiments')
raw_path=Path(r'D:\EEG Experiment Artifacts\2026-11-02\cache\raw_windows_30ch.npy')
raw=np.load(raw_path,mmap_mode='r')
assert raw.shape==(3444,30,30,250)
channels=['Fp1','Fp2','Fz','F3','F4','F7','F8','FC1','FC2','FC5','FC6','Cz','C3','C4','T7','T8','CP1','CP2','CP5','CP6','Pz','P3','P4','P7','P8','PO3','PO4','Oz','O1','O2']
regions=[['Fp1','Fp2','Fz','F3','F4','F7','F8','FC1','FC2','FC5','FC6'],['Cz','C3','C4'],['T7','T8'],['CP1','CP2','CP5','CP6','Pz','P3','P4','P7','P8'],['PO3','PO4','Oz','O1','O2']]
groups=[[channels.index(c) for c in g] for g in regions]
selected=list(range(12))+list(range(16,28))
outfile=root/'plv_35_nodes.npy'; progress_file=root/'plv_progress.json';done_file=root/'plv_complete.json'
if done_file.exists():
 info=json.loads(done_file.read_text(encoding='utf8'));assert info['shape']==[2952,35,35];print('PLV cache already complete',flush=True)
else:
 sos_filters=[butter(4,[lo,hi],btype='bandpass',fs=250,output='sos') for lo,hi in [(1,4),(4,8),(8,14),(14,30),(30,47)]]
 if outfile.exists():
  assert progress_file.exists(),'Incomplete PLV cache has no progress record'
  progress=json.loads(progress_file.read_text(encoding='utf8'));start=progress['next_trial'];out=np.lib.format.open_memmap(outfile,mode='r+')
 else:
  start=0;out=np.lib.format.open_memmap(outfile,mode='w+',dtype='float32',shape=(2952,35,35));progress_file.write_text(json.dumps({'next_trial':0}),encoding='utf8')
 assert out.shape==(2952,35,35)
 started=time.perf_counter()
 for j in range(start,2952):
  x=np.asarray(raw[(j//24)*28+selected[j%24]],dtype=np.float64).transpose(1,0,2).reshape(30,7500)
  region=np.stack([x[ind].mean(axis=0) for ind in groups]);x=np.concatenate([x,region],axis=0)
  total=np.zeros((35,35),dtype=np.float64)
  for sos in sos_filters:
   analytic=hilbert(sosfiltfilt(sos,x,axis=1),axis=1)
   phase=analytic/np.maximum(np.abs(analytic),1e-12)
   total+=np.abs(phase@phase.conj().T/phase.shape[1])
  out[j]=total/len(sos_filters)
  if (j+1)%16==0 or j==2951:
   out.flush();temp=progress_file.with_suffix('.tmp');temp.write_text(json.dumps({'next_trial':j+1}),encoding='utf8');temp.replace(progress_file)
   print('PLV',j+1,'/',2952,'seconds',round(time.perf_counter()-started,1),flush=True)
 digest=hashlib.sha256()
 with raw_path.open('rb') as f:
  for block in iter(lambda:f.read(8*1024*1024),b''):digest.update(block)
 info={'shape':[2952,35,35],'source':str(raw_path),'source_sha256':digest.hexdigest(),'selected_zero_based_videos':selected,'regions':regions,'bands_hz':[[1,4],[4,8],[8,14],[14,30],[30,47]],'method':'fourth-order Butterworth, zero-phase filtering, Hilbert analytic phase, mean PLV over complete 30-second input trial and five bands; regions average normalized channels before PLV','seconds':time.perf_counter()-started}
 done_file.write_text(json.dumps(info,indent=2),encoding='utf8');print('PLV COMPLETE',flush=True)


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import copy
import hashlib
import json
import random
import pickle
import time
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.stats import sem
from scipy.special import expit
from sklearn.metrics import roc_auc_score
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import GATConv, global_mean_pool

torch.set_num_threads(4)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2), encoding='utf-8')

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()


In [ ]:
variant = 'de'
dataset_root = Path(r'C:\FACED dataset')
experiment_root = Path(r'D:\EEG Experiment Artifacts\2026-11-02\binary_de_psd')
run_dir = experiment_root / variant
run_dir.mkdir(parents=True, exist_ok=True)
run_training = True
seed = 42
epochs = 20
batch_size = 32
learning_rate = 0.001
num_classes = 2
class_names = ['Negative', 'Positive']
selected_videos = np.r_[0:12, 16:28]
feature_masks = {'de': [1]*5 + [0]*5, 'psd': [0]*5 + [1]*5, 'de_psd': [1]*10}
channels = ['Fp1','Fp2','Fz','F3','F4','F7','F8','FC1','FC2','FC5','FC6',
            'Cz','C3','C4','T7','T8','CP1','CP2','CP5','CP6','Pz','P3','P4',
            'P7','P8','PO3','PO4','Oz','O1','O2']
settings = {
    'seed': seed, 'epochs': epochs, 'batch_size': batch_size, 'learning_rate': learning_rate,
    'classes': class_names, 'channels': channels, 'selected_video_ids': (selected_videos+1).tolist(),
    'bands': [[1,4],[4,8],[8,14],[14,30],[30,47]],
    'features': 'official FACED DE and PSD files; natural log of PSD, floor 1e-20',
    'psd_definition': 'released FACED FFT/Hann features with globally normalized spectral magnitudes; not a new Welch estimate',
    'scaling': 'per channel/feature mean and std from training subjects only; floor std 1e-6',
    'input': 'ten fixed slots: five DE then five log-PSD; inactive slots masked to zero after scaling',
    'outer_folds': 10, 'inner_validation_fraction': 0.2,
    'labels': 'fixed negative/positive stimulus categories; four neutral videos excluded',
    'evaluation': 'one prediction per second, 30 seconds per trial',
    'selection': 'best validation balanced accuracy over 20 epochs; test evaluated afterwards',
    'graph': 'colin27_1020, distance < 0.15 m, 30 EEG channels',
    'cuda_scatter_may_be_nondeterministic': True
}
print('Variant:', variant, '| Device:', device)
print('Binary stimulus labels: 12 negative and 12 positive videos per subject; no neutral trials.')
print('Screening uses fold 0 only, with subject-disjoint validation and a fixed seed.')
print('PSD means the released FACED features. Log-PSD is used; scalers are fit on training subjects only.')


In [ ]:
cache_dir = experiment_root / 'cache'
cache_dir.mkdir(exist_ok=True)
feature_path = cache_dir / 'features.npy'
cache_info_path = cache_dir / 'features.json'
files = {kind: [dataset_root/'EEG_Features'/kind/f'sub{s:03d}.pkl.pkl' for s in range(123)]
         for kind in ['DE','PSD']}
input_hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest()
                for group in files.values() for p in group}
extractor_path = dataset_root / 'EEG_Features/features_extract.py'
input_hashes[str(extractor_path)] = hashlib.sha256(extractor_path.read_bytes()).hexdigest()

if cache_info_path.exists():
    cache_info = json.loads(cache_info_path.read_text())
    assert cache_info['input_hashes'] == input_hashes
    assert cache_info['settings'] == {k: settings[k] for k in ['features','selected_video_ids','bands']}
else:
    features = np.empty((123,24,30,30,10), dtype=np.float32)
    for s in range(123):
        for kind, columns in [('DE', slice(0,5)), ('PSD', slice(5,10))]:
            with files[kind][s].open('rb') as f:
                values = np.asarray(pickle.load(f))
            assert values.shape == (28,32,30,5) and np.isfinite(values).all()
            values = values[selected_videos,:30].transpose(0,2,1,3)
            if kind == 'PSD':
                assert (values >= 0).all()
                values = np.log(np.maximum(values,1e-20))
            features[s,...,columns] = values
    features = features.reshape(2952,30,30,10)
    assert np.isfinite(features).all()
    np.save(feature_path, features)
    cache_info = {'input_hashes': input_hashes, 'shape': list(features.shape),
                  'settings': {k: settings[k] for k in ['features','selected_video_ids','bands']},
                  'cache_sha256': hashlib.sha256(feature_path.read_bytes()).hexdigest()}
    save_json(cache_info_path, cache_info)
    del features

features = np.load(feature_path, mmap_mode='r')
assert features.shape == (2952,30,30,10)
subjects = np.repeat(np.arange(123),24)
videos = np.tile(selected_videos+1,123)
video_labels = np.r_[np.zeros(12,dtype=int),np.ones(12,dtype=int)]
labels = np.tile(video_labels,123)
splits = []
for fold in range(10):
    test = np.arange(fold*12,(fold+1)*12 if fold<9 else 123)
    remaining = np.setdiff1d(np.arange(123),test)
    shuffled = np.random.default_rng(seed+fold).permutation(remaining)
    n_val = round(0.2*len(remaining))
    val, train = np.sort(shuffled[:n_val]), np.sort(shuffled[n_val:])
    assert not (set(train)&set(val) or set(train)&set(test) or set(val)&set(test))
    splits.append({'fold':fold,'train':train.tolist(),'val':val.tolist(),'test':test.tolist()})
assert sorted(s for f in splits for s in f['test']) == list(range(123))
protocol = {'settings':settings,'splits':splits,'input_hashes':input_hashes,
            'cache_sha256':cache_info['cache_sha256']}
protocol_path = experiment_root / 'protocol.json'
if protocol_path.exists():
    assert json.loads(protocol_path.read_text()) == protocol
else:
    save_json(protocol_path,protocol)
protocol_hash = hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()

positions = mne.channels.make_standard_montage('colin27_1020').get_positions()['ch_pos']
positions = np.stack([positions[c] for c in channels])
adjacency = cdist(positions,positions)<0.15
np.fill_diagonal(adjacency,False)
edge_index = torch.tensor(np.array(adjacency.nonzero()),dtype=torch.long)

def trial_indices(people):
    return np.flatnonzero(np.isin(subjects,people))

def fit_scaler(split):
    directory = experiment_root / 'scalers'
    directory.mkdir(exist_ok=True)
    path = directory / f"fold_{split['fold']:02d}.npz"
    if path.exists():
        saved = np.load(path)
        assert saved['protocol_hash'].item() == protocol_hash
        assert np.array_equal(saved['training_subjects'],split['train'])
        return saved['mean'],saved['std']
    train_values = np.asarray(features[trial_indices(split['train'])])
    mean = train_values.mean(axis=(0,1),dtype=np.float64)
    std = np.maximum(train_values.std(axis=(0,1),dtype=np.float64),1e-6)
    mean,std = mean.astype(np.float32),std.astype(np.float32)
    np.savez(path,mean=mean,std=std,training_subjects=split['train'],protocol_hash=protocol_hash)
    return mean,std

class Trials(Dataset):
    def __init__(self,indices,scaler):
        self.indices = np.asarray(indices)
        self.mean,self.std = scaler
    def __len__(self):
        return len(self.indices)
    def __getitem__(self,index):
        j = self.indices[index]
        values = (features[j].copy()-self.mean)/self.std
        return torch.from_numpy(values),int(labels[j])

def make_loader(indices,scaler,shuffle=False,loader_seed=42):
    return DataLoader(Trials(indices,scaler),batch_size=batch_size,shuffle=shuffle,
                      generator=torch.Generator().manual_seed(loader_seed),num_workers=0,
                      pin_memory=device.type=='cuda')

print('Trials:',len(labels),'| One-second targets:',len(labels)*30)
print('Negative/positive trial counts:',np.bincount(labels))
print('Inputs: DE slots 0-4, log-PSD slots 5-9; graph edges:',edge_index.shape[1])


In [ ]:
class GATLSTM(nn.Module):
    def __init__(self,variant,init_seed):
        super().__init__()
        with torch.random.fork_rng():
            torch.manual_seed(init_seed)
            self.encoder = nn.Sequential(nn.Linear(10,32),nn.GELU(),nn.LayerNorm(32))
            self.gat1 = GATConv(32,16,heads=4,dropout=0.2)
            self.gat2 = GATConv(64,16,heads=1,dropout=0.2)
            self.lstm = nn.LSTM(16,64,batch_first=True)
            self.fc = nn.Linear(64,2)
        self.register_buffer('feature_mask',torch.tensor(feature_masks[variant],dtype=torch.float32))
        self.register_buffer('base_edges',edge_index.clone())
        self.graph_cache = {}

    def forward(self,x):
        b,t,n,d = x.shape
        h = self.encoder((x*self.feature_mask).reshape(b*t*n,d))
        count = b*t
        key = (count,x.device)
        if key not in self.graph_cache:
            offsets = torch.arange(count,device=x.device)*n
            edges = (self.base_edges[:,None,:]+offsets[None,:,None]).reshape(2,-1)
            membership = torch.arange(count,device=x.device).repeat_interleave(n)
            self.graph_cache[key] = edges,membership
        edges,membership = self.graph_cache[key]
        h = F.elu(self.gat1(h,edges))
        h = F.elu(self.gat2(h,edges))
        h = global_mean_pool(h,membership,size=count).reshape(b,t,16)
        h,_ = self.lstm(h)
        return self.fc(h)


In [ ]:
def metrics_from_cm(cm):
    cm = np.asarray(cm, dtype=np.float64)
    true = cm.sum(axis=1)
    predicted = cm.sum(axis=0)
    diagonal = np.diag(cm)
    return {'accuracy': float(diagonal.sum() / cm.sum()),
            'balanced_accuracy': float(np.mean(diagonal / np.maximum(true, 1))),
            'macro_f1': float(np.mean(2 * diagonal / np.maximum(true + predicted, 1)))}

def pass_epoch(model, loader, optimizer=None, collect=False):
    training = optimizer is not None
    model.train(training)
    cm = torch.zeros(num_classes ** 2, device=device, dtype=torch.long)
    loss_sum, total = 0.0, 0
    predictions, logits_list = [], []
    with torch.set_grad_enabled(training):
        for x, y, trial_ids in loader:
            x, y = x.to(device), y.to(device)
            target = y[:, None].expand(-1, x.shape[1])
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(x,trial_ids)
            loss = F.cross_entropy(logits.reshape(-1, num_classes), target.reshape(-1))
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite loss')
            if training:
                loss.backward()
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if not torch.isfinite(norm):
                    raise FloatingPointError('Nonfinite gradient')
                optimizer.step()
            predicted = logits.argmax(dim=-1)
            cm += torch.bincount((target * num_classes + predicted).reshape(-1), minlength=num_classes ** 2)
            total += target.numel()
            loss_sum += loss.item() * target.numel()
            if collect:
                predictions.append(predicted.cpu().numpy())
                logits_list.append(logits.cpu().numpy())
    matrix = cm.reshape(num_classes, num_classes).cpu().numpy()
    result = {'loss': loss_sum / total, **metrics_from_cm(matrix)}
    return result, matrix, (np.concatenate(predictions) if collect else None), (np.concatenate(logits_list) if collect else None)


In [ ]:

# Screening: one fixed subject split, one seed and ten epochs.
epochs = 10
split = splits[0]
assert not (set(split['train']) & set(split['val']) or set(split['train']) & set(split['test']) or set(split['val']) & set(split['test']))
scaler = fit_scaler(split)
train_idx, val_idx = [trial_indices(split[k]) for k in ['train', 'val']]
output_root = Path(r'D:\EEG Experiment Artifacts\2026-11-02\graph_experiments')
output_root.mkdir(parents=True, exist_ok=True)
reference_edges = edge_index.clone()
threshold = 0.5530934343434344  # Fold 0 DE SVM, validation balanced accuracy.
from torch_geometric.utils import softmax
class WeightedGAT(GATConv):
    def edge_update(self,alpha_j,alpha_i,edge_attr,index,ptr,dim_size):
        alpha=alpha_j if alpha_i is None else alpha_j+alpha_i
        if index.numel()==0:return alpha
        alpha=F.leaky_relu(alpha,self.negative_slope)
        if edge_attr is not None: alpha=alpha+torch.log(edge_attr.reshape(-1,1).clamp_min(1e-8))
        return F.dropout(softmax(alpha,index,ptr,dim_size),p=self.dropout,training=self.training)
plv_file=output_root/'plv_35_nodes.npy'
plv_info=json.loads((output_root/'plv_complete.json').read_text(encoding='utf8'))
plv=np.load(plv_file,mmap_mode='r')
assert plv.shape==(2952,35,35)
similarity_file=output_root/'training_de_similarity_fold00.npy'
if similarity_file.exists(): similarity=np.load(similarity_file)
else:
    similarity_sum=np.zeros((30,30),dtype=np.float64)
    for j in train_idx:
        v=np.asarray(features[j,:,:,:5],dtype=np.float64)
        v=v-v.mean(axis=0,keepdims=True)
        v=v/np.maximum(np.linalg.norm(v,axis=0,keepdims=True),1e-12)
        similarity_sum+=np.abs(np.einsum('tif,tjf->ijf',v,v)).mean(axis=-1)
    similarity=(similarity_sum/len(train_idx)).astype(np.float32)
    np.save(similarity_file,similarity)
def top4(values):
    v=np.array(values,copy=True);np.fill_diagonal(v,-np.inf)
    idx=np.argsort(-v,axis=1)[:,:4]
    adj=np.zeros((30,30),dtype=bool);adj[np.arange(30)[:,None],idx]=True
    return adj|adj.T
distance=cdist(positions,positions);np.fill_diagonal(distance,np.inf)
spatial_knn=top4(-distance)
similarity_graph=top4(similarity)
ref_adjacency=np.zeros((30,30),dtype=bool)
ref_adjacency[tuple(reference_edges.numpy())]=True
class IndexedTrials(Trials):
    def __getitem__(self,index):
        x,y=super().__getitem__(index)
        return x,y,int(self.indices[index])
Trials=IndexedTrials
class FunctionalGraph(GATLSTM):
    def __init__(self,variant,init_seed,kind):
        super().__init__(variant,init_seed)
        self.kind=kind
        if kind=='plv_weights':
            first_state=self.gat1.state_dict();second_state=self.gat2.state_dict()
            with torch.random.fork_rng():
                torch.manual_seed(init_seed)
                self.gat1=WeightedGAT(32,16,heads=4,dropout=0.2,fill_value=1.0)
                self.gat2=WeightedGAT(64,16,heads=1,dropout=0.2,fill_value=1.0)
            self.gat1.load_state_dict(first_state);self.gat2.load_state_dict(second_state)
    def forward(self,x,trial_ids):
        b,t,n,d=x.shape
        h=self.encoder((x*self.feature_mask).reshape(b*t*n,d))
        edges=[];weights=[]
        for i,j in enumerate(trial_ids.tolist()):
            conn=np.asarray(plv[j,:30,:30])
            if self.kind=='de_graph': adj=similarity_graph
            elif self.kind=='plv_graph': adj=top4(conn)
            elif self.kind=='mixed_graph': adj=spatial_knn|top4(conn)
            elif self.kind=='plv_weights': adj=ref_adjacency
            else: raise ValueError(self.kind)
            src,dst=np.nonzero(adj);local=np.stack([src,dst])
            w=conn[src,dst] if self.kind=='plv_weights' else None
            for sec in range(t):
                edges.append(local+(i*t+sec)*n)
                if w is not None:weights.append(w)
        edge_tensor=torch.as_tensor(np.concatenate(edges,axis=1),device=x.device,dtype=torch.long)
        if weights:
            weight_tensor=torch.as_tensor(np.concatenate(weights),device=x.device,dtype=x.dtype)
            h=F.elu(self.gat1(h,edge_tensor,edge_attr=weight_tensor));h=F.elu(self.gat2(h,edge_tensor,edge_attr=weight_tensor))
        else:
            h=F.elu(self.gat1(h,edge_tensor));h=F.elu(self.gat2(h,edge_tensor))
        membership=torch.arange(b*t,device=x.device).repeat_interleave(n)
        pooled=global_mean_pool(h,membership,size=b*t).reshape(b,t,16)
        pooled,_=self.lstm(pooled)
        return self.fc(pooled)
configs=[('A00012','de_graph',reference_edges),('A00013','plv_graph',reference_edges),('A00014','mixed_graph',reference_edges),('A00017','plv_weights',reference_edges)]

results = []
for experiment_id, graph_name, chosen_edges in configs:
    folder = output_root / 'runs' / experiment_id
    folder.mkdir(parents=True, exist_ok=True)
    manifest = {'id': experiment_id, 'graph': graph_name, 'fold': 0, 'seed': seed,
                'epochs': epochs, 'threshold_val_balanced_accuracy': threshold,
                'protocol_hash': protocol_hash, 'cache_sha256': cache_info['cache_sha256'],
                'train_subjects': split['train'], 'validation_subjects': split['val'],
                'test_subjects': split['test'], 'edge_count_without_self_loops': chosen_edges.shape[1],
                'device': str(device), 'torch': str(torch.__version__),
                'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None}
    config_file = folder / 'config.json'
    if config_file.exists():
        assert json.loads(config_file.read_text(encoding='utf8')) == manifest
    else:
        save_json(config_file, manifest)
    if (folder / 'metrics.json').exists():
        assert (folder/'best_model.pt').exists() and (folder/'validation_predictions.npz').exists()
        results.append(json.loads((folder/'metrics.json').read_text(encoding='utf8')))
        print(experiment_id, 'already completed', flush=True)
        continue
    edge_index = chosen_edges
    seed_all(seed)
    train_loader = make_loader(train_idx,scaler,True,seed)
    val_loader = make_loader(val_idx,scaler)
    model = FunctionalGraph('de',seed,graph_name).to(device)
    optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
    best_score = -1.0
    history = []
    started = time.perf_counter()
    for epoch in range(1,epochs+1):
        start_epoch=time.perf_counter()
        train_metrics,_,_,_ = pass_epoch(model,train_loader,optimizer)
        val_metrics,_,_,_ = pass_epoch(model,val_loader)
        history.append({'epoch':epoch,'seconds':time.perf_counter()-start_epoch,
                        **{'train_'+k:v for k,v in train_metrics.items()},
                        **{'validation_'+k:v for k,v in val_metrics.items()}})
        pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
        if val_metrics['balanced_accuracy']>best_score:
            best_score=val_metrics['balanced_accuracy']
            torch.save({'state_dict':model.state_dict(),'epoch':epoch,'validation':val_metrics},folder/'best_model.pt')
        print(f"{experiment_id} epoch {epoch}/{epochs} val balanced accuracy={val_metrics['balanced_accuracy']:.4f}",flush=True)
    checkpoint=torch.load(folder/'best_model.pt',map_location=device,weights_only=False)
    model.load_state_dict(checkpoint['state_dict'])
    val_metrics,matrix,predicted,logits=pass_epoch(model,val_loader,collect=True)
    assert np.isclose(val_metrics['balanced_accuracy'],best_score,atol=1e-6)
    target=np.repeat(labels[val_idx,None],30,axis=1)
    np.savez_compressed(folder/'validation_predictions.npz',predictions=predicted,logits=logits,
                        labels=target,subjects=subjects[val_idx],videos=videos[val_idx])
    result={'id':experiment_id,'graph':graph_name,'best_epoch':checkpoint['epoch'],
            'validation':val_metrics,'validation_confusion_matrix':matrix.tolist(),
            'threshold':threshold,'passes_threshold':bool(best_score>=threshold),
            'seconds':time.perf_counter()-started,'parameters':sum(p.numel() for p in model.parameters())}
    save_json(folder/'metrics.json',result)
    results.append(result)
    print('COMPLETE',json.dumps(result),flush=True)
    del model,optimizer
    if device.type=='cuda': torch.cuda.empty_cache()
pd.DataFrame([{'id':r['id'],'graph':r['graph'],'best_epoch':r['best_epoch'],
               'validation_balanced_accuracy':r['validation']['balanced_accuracy'],
               'seconds':r['seconds']} for r in results])
